# 🧪 Bio-JEPA — Baselines Reviewer : GraphMAE & MolCLR

**Objectif** : Répondre au reviewer qui demande une comparaison avec GraphMAE et MolCLR.  
**Protocole identique Bio-JEPA** : ZINC250k 100ep, sonde MLP gelée, 5 runs, N ∈ {10,50,100,200,500,1000}, cible A2A (ChEMBL251).

| Modèle | Type SSL | Pré-entraînement |
|---|---|---|
| **GraphMAE** | Masking latent (MAE sur features) | ZINC250k 100ep |
| **MolCLR** | Contrastif (augmentation moléculaire) | ZINC250k 100ep |
| Bio-JEPA (référence) | Prédiction latente (JEPA) | ZINC250k 100ep |

## 1. Installation & Imports

In [1]:
# Installation
!pip install torch-geometric torch-scatter torch-sparse -q
!pip install rdkit -q

# Monter le Drive
from google.colab import drive
drive.mount('/content/drive')

import os, json, torch, random
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINEConv, global_mean_pool
from torch_geometric.data import DataLoader
from scipy.stats import pearsonr

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')

# Paths Drive
CKPT_DIR    = '/content/drive/MyDrive/Bio-JEPA-checkpoints'
RESULTS_DIR = '/content/drive/MyDrive/Bio-JEPA-results'
os.makedirs(RESULTS_DIR, exist_ok=True)

# Cloner le repo Bio-JEPA (pour réutiliser mol_graph.py)
if not os.path.exists('/content/Bio-JEPA'):
    !git clone https://github.com/7Nayy/Bio-JEPA /content/Bio-JEPA
import sys
sys.path.insert(0, '/content/Bio-JEPA')

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
Mounted at /content/drive


ModuleNotFoundError: No module named 'torch_geometric'

## 2. Chargement des datasets (depuis le Drive)

In [ ]:
# Charger les graphes déjà construits (même pipeline que Bio-JEPA)
zinc_graphs    = torch.load(f'{CKPT_DIR}/zinc_graphs.pt', weights_only=False)
chembl251_graphs = torch.load(f'{CKPT_DIR}/chembl251_graphs.pt', weights_only=False)

print(f'ZINC250k   : {len(zinc_graphs)} molécules')
print(f'ChEMBL251  : {len(chembl251_graphs)} molécules')
print(f'Node features : {zinc_graphs[0].x.shape[1]}')
print(f'Edge features : {zinc_graphs[0].edge_attr.shape[1]}')

NODE_DIM = zinc_graphs[0].x.shape[1]   # 29
EDGE_DIM = zinc_graphs[0].edge_attr.shape[1]  # 7
HIDDEN   = 256
LATENT   = 256

## 3. Architecture GraphMAE

In [ ]:
# ============================================================
# GraphMAE — Hou et al. (NeurIPS 2022)
# Principe : masquer des nœuds, encoder, reconstruire les
# features originaux avec une perte cosinus (scaled cosine error).
# Différence clé vs Bio-JEPA : reconstruction dans l'espace INPUT
# (features atomiques bruts) vs prédiction dans l'espace LATENT.
# ============================================================

class GraphMAEEncoder(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden, latent, num_layers=5):
        super().__init__()
        self.input_proj = nn.Linear(node_dim, hidden)
        self.mask_token  = nn.Parameter(torch.zeros(hidden))  # token de masque appris

        self.convs = nn.ModuleList()
        self.bns   = nn.ModuleList()
        for _ in range(num_layers):
            mlp = nn.Sequential(nn.Linear(hidden, hidden*2), nn.ReLU(), nn.Linear(hidden*2, hidden))
            self.convs.append(GINEConv(mlp, edge_dim=edge_dim))
            self.bns.append(nn.BatchNorm1d(hidden))

        self.output_proj = nn.Linear(hidden, latent)

    def forward(self, x, edge_index, edge_attr, mask=None):
        h = self.input_proj(x)
        if mask is not None:
            h[mask] = self.mask_token  # remplace les nœuds masqués
        for conv, bn in zip(self.convs, self.bns):
            h = F.relu(bn(conv(h, edge_index, edge_attr)))
        return self.output_proj(h)


class GraphMAEDecoder(nn.Module):
    """Decoder léger : reconstruire les features originaux."""
    def __init__(self, latent, hidden, node_dim, num_layers=1):
        super().__init__()
        layers = []
        in_dim = latent
        for _ in range(num_layers):
            layers += [nn.Linear(in_dim, hidden), nn.ReLU()]
            in_dim = hidden
        layers.append(nn.Linear(in_dim, node_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, z):
        return self.net(z)


class GraphMAE(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden=256, latent=256,
                 mask_ratio=0.15, num_layers=5):
        super().__init__()
        self.encoder  = GraphMAEEncoder(node_dim, edge_dim, hidden, latent, num_layers)
        self.decoder  = GraphMAEDecoder(latent, hidden, node_dim, num_layers=1)
        self.mask_ratio = mask_ratio

    def forward(self, data):
        x, ei, ea, batch = data.x, data.edge_index, data.edge_attr, data.batch

        # Sélection aléatoire des nœuds à masquer
        N = x.size(0)
        num_mask = max(1, int(N * self.mask_ratio))
        mask_idx = torch.randperm(N, device=x.device)[:num_mask]
        mask = torch.zeros(N, dtype=torch.bool, device=x.device)
        mask[mask_idx] = True

        # Encoder avec nœuds masqués
        z = self.encoder(x, ei, ea, mask=mask)

        # Reconstruire uniquement les nœuds masqués
        z_masked = z[mask]
        x_pred   = self.decoder(z_masked)
        x_true   = x[mask]

        # Scaled Cosine Error (GraphMAE original)
        loss = 1 - F.cosine_similarity(x_pred, x_true, dim=-1).mean()
        return loss

    def encode_graph(self, data):
        """Embedding niveau graphe (pas de masque, pour évaluation)."""
        z = self.encoder(data.x, data.edge_index, data.edge_attr)
        return global_mean_pool(z, data.batch)


print('GraphMAE défini ✓')

## 4. Architecture MolCLR

In [ ]:
# ============================================================
# MolCLR — Wang et al. (Nature Machine Intelligence 2022)
# Principe : augmentation de graphes moléculaires (suppression
# d'atomes/liaisons, masquage de sous-graphes) + perte
# contrastive NT-Xent sur les représentations.
# ============================================================

def augment_molecule(data, aug_ratio=0.2):
    """Augmentation MolCLR : suppression aléatoire de nœuds."""
    import copy
    data2 = copy.deepcopy(data)
    N = data2.x.size(0)
    num_drop = max(1, int(N * aug_ratio))
    drop_idx = torch.randperm(N)[:num_drop]
    keep = torch.ones(N, dtype=torch.bool)
    keep[drop_idx] = False

    # Reconstruire le graphe sans les nœuds supprimés
    new_idx = torch.full((N,), -1, dtype=torch.long)
    new_idx[keep] = torch.arange(keep.sum())

    # Filtrer les arêtes
    src, dst = data2.edge_index
    edge_mask = keep[src] & keep[dst]
    data2.edge_index = new_idx[data2.edge_index[:, edge_mask]]
    data2.edge_attr  = data2.edge_attr[edge_mask]
    data2.x = data2.x[keep]
    return data2


class MolCLREncoder(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden=256, latent=256, num_layers=5):
        super().__init__()
        self.input_proj = nn.Linear(node_dim, hidden)
        self.convs = nn.ModuleList()
        self.bns   = nn.ModuleList()
        for _ in range(num_layers):
            mlp = nn.Sequential(nn.Linear(hidden, hidden*2), nn.ReLU(), nn.Linear(hidden*2, hidden))
            self.convs.append(GINEConv(mlp, edge_dim=edge_dim))
            self.bns.append(nn.BatchNorm1d(hidden))
        self.pool = global_mean_pool
        self.proj = nn.Sequential(
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, latent)
        )

    def forward(self, x, edge_index, edge_attr, batch):
        h = self.input_proj(x)
        for conv, bn in zip(self.convs, self.bns):
            h = F.relu(bn(conv(h, edge_index, edge_attr)))
        g = self.pool(h, batch)
        return self.proj(g)  # projection head inclus

    def encode_graph(self, data):
        h = self.input_proj(data.x)
        for conv, bn in zip(self.convs, self.bns):
            h = F.relu(bn(conv(h, data.edge_index, data.edge_attr)))
        return self.pool(h, data.batch)  # sans projection pour évaluation


def nt_xent_loss(z1, z2, temperature=0.1):
    """NT-Xent loss (SimCLR) pour MolCLR."""
    B = z1.size(0)
    z = F.normalize(torch.cat([z1, z2], dim=0), dim=1)
    sim = torch.mm(z, z.t()) / temperature
    # Masquer la diagonale
    mask = torch.eye(2*B, device=z.device, dtype=torch.bool)
    sim.masked_fill_(mask, -1e9)
    # Labels : pour i, le positif est i+B
    labels = torch.cat([torch.arange(B, 2*B), torch.arange(0, B)]).to(z.device)
    return F.cross_entropy(sim, labels)


print('MolCLR défini ✓')

## 5. Pré-entraînement GraphMAE (ZINC250k, 100 époques)

In [ ]:
def pretrain_graphmae(graphs, epochs=100, batch_size=256, lr=1e-3, save_path=None):
    loader = DataLoader(graphs, batch_size=batch_size, shuffle=True, num_workers=2)
    model  = GraphMAE(NODE_DIM, EDGE_DIM, HIDDEN, LATENT, mask_ratio=0.15).to(DEVICE)
    opt    = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    print(f'GraphMAE params : {sum(p.numel() for p in model.parameters()):,}')

    best_loss = float('inf')
    for ep in range(1, epochs+1):
        model.train()
        total = 0
        for batch in loader:
            batch = batch.to(DEVICE)
            loss  = model(batch)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item()
        sched.step()
        avg = total / len(loader)
        if ep % 10 == 0:
            print(f'  Epoch {ep:3d}/{epochs} | Loss {avg:.6f}')
        if avg < best_loss:
            best_loss = avg
            if save_path:
                torch.save(model.state_dict(), save_path)

    print(f'\n✅ GraphMAE pré-entraîné | Best loss = {best_loss:.6f}')
    return model


GRAPHMAE_CKPT = f'{CKPT_DIR}/graphmae_pretrained.pt'

if os.path.exists(GRAPHMAE_CKPT):
    print('Checkpoint GraphMAE trouvé, chargement...')
    graphmae = GraphMAE(NODE_DIM, EDGE_DIM, HIDDEN, LATENT).to(DEVICE)
    graphmae.load_state_dict(torch.load(GRAPHMAE_CKPT, map_location=DEVICE, weights_only=False))
else:
    print('Pré-entraînement GraphMAE sur ZINC250k (100 ep)...')
    graphmae = pretrain_graphmae(zinc_graphs, epochs=100, save_path=GRAPHMAE_CKPT)

graphmae.eval()
print('GraphMAE prêt ✓')

## 6. Pré-entraînement MolCLR (ZINC250k, 100 époques)

In [ ]:
def pretrain_molclr(graphs, epochs=100, batch_size=256, lr=1e-3, save_path=None):
    loader = DataLoader(graphs, batch_size=batch_size, shuffle=True, num_workers=2)
    model  = MolCLREncoder(NODE_DIM, EDGE_DIM, HIDDEN, LATENT).to(DEVICE)
    opt    = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    print(f'MolCLR params : {sum(p.numel() for p in model.parameters()):,}')

    best_loss = float('inf')
    for ep in range(1, epochs+1):
        model.train()
        total = 0
        for batch in loader:
            # Deux vues augmentées par batch
            from torch_geometric.data import Batch
            aug1 = Batch.from_data_list([augment_molecule(d) for d in batch.to_data_list()]).to(DEVICE)
            aug2 = Batch.from_data_list([augment_molecule(d) for d in batch.to_data_list()]).to(DEVICE)

            z1 = model(aug1.x, aug1.edge_index, aug1.edge_attr, aug1.batch)
            z2 = model(aug2.x, aug2.edge_index, aug2.edge_attr, aug2.batch)
            loss = nt_xent_loss(z1, z2)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item()
        sched.step()
        avg = total / len(loader)
        if ep % 10 == 0:
            print(f'  Epoch {ep:3d}/{epochs} | Loss {avg:.6f}')
        if avg < best_loss:
            best_loss = avg
            if save_path:
                torch.save(model.state_dict(), save_path)

    print(f'\n✅ MolCLR pré-entraîné | Best loss = {best_loss:.6f}')
    return model


MOLCLR_CKPT = f'{CKPT_DIR}/molclr_pretrained.pt'

if os.path.exists(MOLCLR_CKPT):
    print('Checkpoint MolCLR trouvé, chargement...')
    molclr = MolCLREncoder(NODE_DIM, EDGE_DIM, HIDDEN, LATENT).to(DEVICE)
    molclr.load_state_dict(torch.load(MOLCLR_CKPT, map_location=DEVICE, weights_only=False))
else:
    print('Pré-entraînement MolCLR sur ZINC250k (100 ep)...')
    molclr = pretrain_molclr(zinc_graphs, epochs=100, save_path=MOLCLR_CKPT)

molclr.eval()
print('MolCLR prêt ✓')

## 7. Évaluation Few-Shot — Protocole identique Bio-JEPA

In [ ]:
# ============================================================
# Protocole identique Bio-JEPA :
# - Encodeur gelé (frozen)
# - Sonde MLP légère entraînable
# - 5 runs par N, Pearson r sur test
# ============================================================

class MLPProbe(nn.Module):
    def __init__(self, in_dim=256, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden, 1)
        )
    def forward(self, x): return self.net(x).squeeze(-1)


def get_embeddings(model, model_type, graphs, batch_size=512):
    """Extrait les embeddings graphe en mode eval (encodeur gelé)."""
    loader = DataLoader(graphs, batch_size=batch_size, shuffle=False)
    zs, ys = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            if model_type == 'graphmae':
                z = model.encode_graph(batch)
            elif model_type == 'molclr':
                z = model.encode_graph(batch)
            zs.append(z.cpu())
            ys.append(batch.y.cpu())
    return torch.cat(zs), torch.cat(ys)


def fewshot_eval(embeddings, labels, N, n_runs=5, probe_epochs=100, lr=1e-3):
    """Few-shot avec sonde MLP gelée, N exemples d'entraînement."""
    results = []
    for run in range(n_runs):
        torch.manual_seed(run * 42)
        idx   = torch.randperm(len(embeddings))
        train_idx = idx[:N]
        test_idx  = idx[N:N+min(500, len(idx)-N)]  # 500 molécules de test

        X_train = embeddings[train_idx].to(DEVICE)
        y_train = labels[train_idx].float().to(DEVICE)
        X_test  = embeddings[test_idx].to(DEVICE)
        y_test  = labels[test_idx].float().cpu().numpy()

        probe = MLPProbe(in_dim=embeddings.shape[1]).to(DEVICE)
        opt   = torch.optim.Adam(probe.parameters(), lr=lr)

        for _ in range(probe_epochs):
            probe.train()
            loss = F.mse_loss(probe(X_train), y_train)
            opt.zero_grad(); loss.backward(); opt.step()

        probe.eval()
        with torch.no_grad():
            preds = probe(X_test).cpu().numpy()
        r, _ = pearsonr(preds, y_test)
        results.append(r)

    return np.mean(results), np.std(results)


print('Protocole few-shot défini ✓')

## 8. Calcul des embeddings (1 fois, avant les runs)

In [ ]:
print('Calcul des embeddings GraphMAE...')
emb_graphmae, labels = get_embeddings(graphmae, 'graphmae', chembl251_graphs)
print(f'  GraphMAE embeddings : {emb_graphmae.shape}')

print('Calcul des embeddings MolCLR...')
emb_molclr, _ = get_embeddings(molclr, 'molclr', chembl251_graphs)
print(f'  MolCLR embeddings   : {emb_molclr.shape}')

## 9. Few-shot sur tous les N

In [ ]:
N_VALUES = [10, 50, 100, 200, 500, 1000]

results_graphmae = {}
results_molclr   = {}

print('=== GraphMAE Few-Shot ===')
for N in N_VALUES:
    mean, std = fewshot_eval(emb_graphmae, labels, N)
    results_graphmae[N] = {'mean': float(mean), 'std': float(std)}
    print(f'  N={N:4d} | r = {mean:.3f} ± {std:.3f}')

print('\n=== MolCLR Few-Shot ===')
for N in N_VALUES:
    mean, std = fewshot_eval(emb_molclr, labels, N)
    results_molclr[N] = {'mean': float(mean), 'std': float(std)}
    print(f'  N={N:4d} | r = {mean:.3f} ± {std:.3f}')

## 10. Tableau comparatif final

In [ ]:
# Résultats Bio-JEPA & AttrMasking déjà connus
biojepa_known = {10: (0.036, 0.075), 50: (0.130, 0.054), 100: (0.174, 0.068),
                 200: (0.278, 0.054), 500: (0.465, 0.026), 1000: (0.542, 0.020)}
attrmasking_known = {10: (0.120, 0.021), 50: (0.128, 0.011), 100: (0.098, 0.022),
                     200: (0.102, 0.020), 500: (0.117, 0.020), 1000: (0.107, 0.050)}
gnn_sup_known = {10: (0.047, 0.075), 50: (0.338, 0.079), 100: (0.465, 0.049),
                 200: (0.530, 0.023), 500: (0.651, 0.010), 1000: (0.704, 0.004)}

print('\n' + '='*80)
print(f"{'N':>6} | {'Bio-JEPA':>16} | {'GraphMAE':>16} | {'MolCLR':>16} | {'AttrMasking':>16} | {'GNN sup.':>16}")
print('='*80)
for N in N_VALUES:
    bj   = biojepa_known[N]
    gm   = results_graphmae[N]
    mc   = results_molclr[N]
    am   = attrmasking_known[N]
    sup  = gnn_sup_known[N]
    print(f"{N:>6} | {bj[0]:6.3f}±{bj[1]:.3f}  | {gm['mean']:6.3f}±{gm['std']:.3f}  | "
          f"{mc['mean']:6.3f}±{mc['std']:.3f}  | {am[0]:6.3f}±{am[1]:.3f}  | {sup[0]:6.3f}±{sup[1]:.3f}")
print('='*80)

## 11. Sauvegarde des résultats

In [ ]:
all_results = {
    'graphmae_fewshot': results_graphmae,
    'molclr_fewshot':   results_molclr,
    'protocol': {
        'pretrain_dataset': 'ZINC250k',
        'pretrain_epochs': 100,
        'eval_dataset': 'ChEMBL251 (A2A)',
        'n_runs': 5,
        'probe': 'MLP gelée (frozen encoder)',
        'N_values': N_VALUES
    }
}

out_path = f'{RESULTS_DIR}/baselines_graphmae_molclr.json'
with open(out_path, 'w') as f:
    json.dump(all_results, f, indent=2)

print(f'✅ Résultats sauvegardés : {out_path}')
print('\n📌 Prochaine étape : mettre à jour tab:attrmasking dans biojepa_v4.tex avec GraphMAE + MolCLR')

## 12. Template LaTeX à copier dans biojepa_v4.tex

Une fois les résultats obtenus, remplacer le tableau `tab:attrmasking` par cette version étendue :

In [ ]:
# Génération automatique du code LaTeX avec les vrais résultats
print(r'\begin{table}[htbp]')
print(r'\centering')
print(r"\caption{Comparaison few-shot Pearson $r$ --- Bio-JEPA vs baselines SSL (ChEMBL251, A2A)}")
print(r'\label{tab:attrmasking}')
print(r'\resizebox{\columnwidth}{!}{%')
print(r'\begin{tabular}{lcccc}')
print(r'\toprule')
print(r'\textbf{$N$} & \textbf{Bio-JEPA} & \textbf{GraphMAE} & \textbf{MolCLR} & \textbf{AttrMasking} & \textbf{GNN sup.} \\\\')
print(r'\midrule')
for N in N_VALUES:
    bj  = biojepa_known[N]
    gm  = results_graphmae[N]
    mc  = results_molclr[N]
    am  = attrmasking_known[N]
    sup = gnn_sup_known[N]
    print(f"{N:<4} & ${bj[0]:.3f} \\pm {bj[1]:.3f}$ & ${gm['mean']:.3f} \\pm {gm['std']:.3f}$ & "
          f"${mc['mean']:.3f} \\pm {mc['std']:.3f}$ & ${am[0]:.3f} \\pm {am[1]:.3f}$ & ${sup[0]:.3f} \\pm {sup[1]:.3f}$ \\\\")
print(r'\bottomrule')
print(r'\end{tabular}%')
print(r'}')
print(r'\end{table}')